In [1]:
import os
os.chdir('/content')
!unzip -q /content/Boomer.zip

In [2]:
%cd /content/Boomer/

/content/Boomer


In [3]:
!pip install -q -U transformers accelerate diffusers datasets huggingface_hub safetensors webdataset pytest einops ninja packaging wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 184.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.1/509.1 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.2/375.2 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 68.0 MB/s eta 0:00:00


In [4]:
%pip install -q flash-linear-attention einops ninja packaging

%cd /content/Boomer
!python tools/probe_sequence_mixers.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.3/595.3 kB 44.1 MB/s eta 0:00:00
/content/Boomer
local_mamba_repo: missing /content/mamba
local_fla_repo: missing /content/flash-linear-attention
torch=2.10.0+cu128
cuda_available=True
torch_cuda=12.8
cuda_device=NVIDIA RTX PRO 6000 Blackwell Server Edition
probe_device=cuda probe_dtype=torch.bfloat16

== raw imports ==
mamba_ssm: missing (ModuleNotFoundError: No module named 'mamba_ssm')
mamba_ssm.modules.mamba3: missing (ModuleNotFoundError: No module named 'mamba_ssm')
mamba3: missing (ModuleNotFoundError: No module named 'mamba3')
mamba: missing (ModuleNotFoundError: No module named 'mamba')
causal_conv1d: missing (ModuleNotFoundError: No module named 'causal_conv1d')
fla: FOUND 0.5.0
fla.layers: F

In [5]:
# STORK is required for training previews (--preview-sampler stork2) and sample_latent_dit.py.
# Prefer ../STORK (sibling of Boomer) when present; otherwise download to /content/STORK.
from pathlib import Path
import shutil
import urllib.request
import zipfile

BOOMER = Path("/content/Boomer")
STORK_CANDIDATES = [
    BOOMER.resolve().parent / "STORK",
    Path("/content/STORK"),
]

STORK_DIR = next(
    (path for path in STORK_CANDIDATES if (path / "STORKScheduler.py").is_file()),
    Path("/content/STORK"),
)
scheduler = STORK_DIR / "STORKScheduler.py"

if not scheduler.is_file():
    zip_path = Path("/content/stork.zip")
    print("Downloading STORK to /content/STORK ...")
    urllib.request.urlretrieve(
        "https://github.com/ZT220501/STORK/archive/refs/heads/main.zip",
        zip_path,
    )
    if STORK_DIR.exists():
        shutil.rmtree(STORK_DIR)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content")
    shutil.move("/content/STORK-main", STORK_DIR)
    zip_path.unlink(missing_ok=True)
    scheduler = STORK_DIR / "STORKScheduler.py"

assert scheduler.is_file(), f"missing {scheduler}"
print("STORK ready ->", scheduler)

STORK ready -> /content/STORK/STORKScheduler.py


In [6]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

import wandb

try:
    wandb.login(key=userdata.get("WANDB_API_KEY"))
except Exception:
    wandb.login()

In [7]:
# Skip flash-attn-4 / xformers on Blackwell Colab — FA4 JIT fails and xformers conflicts with FA4.
# Use --image-attn-backend sdpa and --cross-attn-backend sdpa for training.
print("Using PyTorch SDPA for image + cross attention.")

Using PyTorch SDPA for image + cross attention.


In [8]:
import torch
import torch.nn.functional as F

# Quick SDPA smoke test (same backend used during training)
q = torch.randn(2, 256, 16, 72, device="cuda", dtype=torch.bfloat16).transpose(1, 2)
k = torch.randn(2, 256, 16, 72, device="cuda", dtype=torch.bfloat16).transpose(1, 2)
v = torch.randn(2, 256, 16, 72, device="cuda", dtype=torch.bfloat16).transpose(1, 2)
out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=False)
print("torch:", torch.__version__, "| sdpa ok:", tuple(out.shape))

torch: 2.10.0+cu128 | sdpa ok: (2, 16, 256, 72)


In [9]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/BoomerMain")
STATS_DIR = DRIVE_ROOT / "stats"
CKPT_DIR = DRIVE_ROOT / "checkpoints" / "boomer_fla_journeydb_rope"
PREVIEW_DIR = DRIVE_ROOT / "previews" / "boomer_fla_journeydb_rope"
PREVIEW_RECORDS_DIR = DRIVE_ROOT / "preview" / "records"
PREVIEW_RECORD = PREVIEW_RECORDS_DIR / "preview_00.pt"
LOG_CSV = DRIVE_ROOT / "logs" / "boomer_fla_journeydb_rope_loss.csv"
WDS_STATS_DIR = Path("/content/wds_stats")
WDS_ROLL_DIR = Path("/content/wds_roll")

for path in [STATS_DIR, CKPT_DIR, PREVIEW_DIR, PREVIEW_RECORDS_DIR, LOG_CSV.parent, WDS_STATS_DIR, WDS_ROLL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

STATS_JSON = STATS_DIR / "journeydb_latent_stats.json"
LATENT_STATS = str(STATS_JSON)

print("stats     ->", STATS_JSON)
print("ckpts     ->", CKPT_DIR)
print("previews  ->", PREVIEW_DIR)
print("preview records ->", PREVIEW_RECORDS_DIR)
print("loss csv  ->", LOG_CSV)

Mounted at /content/drive
stats     -> /content/drive/MyDrive/BoomerMain/stats/journeydb_latent_stats.json
ckpts     -> /content/drive/MyDrive/BoomerMain/checkpoints/boomer_fla_journeydb_rope
previews  -> /content/drive/MyDrive/BoomerMain/previews/boomer_fla_journeydb_rope
preview records -> /content/drive/MyDrive/BoomerMain/preview/records
loss csv  -> /content/drive/MyDrive/BoomerMain/logs/boomer_fla_journeydb_rope_loss.csv


In [ ]:
# Download local shards for stats + preview records (~2.2k samples/shard).
# Run BEFORE the stats cell. HF streaming stats hangs on Colab (1728-shard API listing).
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import hf_hub_download

REPO_ID = "akrao9/JourneyDBCache"
STATS_SHARD_COUNT = 22  # ~50k samples, ~48GB
WDS_DIR = WDS_STATS_DIR

def download_shard(index: int) -> str:
    filename = f"train/train-{index:06d}.tar"
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=filename,
        repo_type="dataset",
        local_dir=str(WDS_DIR),
    )
    print("downloaded", filename, flush=True)
    return path

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(download_shard, i) for i in range(STATS_SHARD_COUNT)]
    for future in as_completed(futures):
        future.result()

print(
    "done ->",
    WDS_DIR / "train",
    "| shards:",
    len(list((WDS_DIR / "train").glob("*.tar"))),
    flush=True,
)

In [ ]:
%cd /content/Boomer

# Channel stats from local shards (fast). Run the shard-download cell first.
# HF streaming hangs silently on Colab while listing ~1728 remote shards.
import os
import subprocess
import sys
from pathlib import Path

RECOMPUTE_STATS = True
STATS_MAX_RECORDS = 50_000
LOCAL_WDS_GLOB = str(WDS_STATS_DIR / "train" / "*.tar")
local_shards = sorted(WDS_STATS_DIR.glob("train/*.tar"))

if Path(LATENT_STATS).exists() and not RECOMPUTE_STATS:
    print("stats already exist ->", LATENT_STATS)
elif not local_shards:
    raise FileNotFoundError(
        f"No local shards under {WDS_STATS_DIR}/train — run the download cell first."
    )
else:
    est_samples = len(local_shards) * 2_200
    target = min(STATS_MAX_RECORDS, est_samples)
    print(f"local shards={len(local_shards)} target_records={target:,} -> {LATENT_STATS}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    subprocess.run(
        [
            sys.executable,
            "-u",
            "tools/inspect_latent_stats.py",
            "--cache-wds",
            LOCAL_WDS_GLOB,
            "--max-records",
            str(target),
            "--log-every",
            "2000",
            "--json-out",
            LATENT_STATS,
        ],
        check=True,
        env=env,
    )

In [ ]:
# Build 5 fixed preview .pt records on Drive (required for --preview-every with WDS training).
%cd /content/Boomer

import torch
import webdataset as wds
from boomer.datasets.wds_latents import sample_to_record

REBUILD_PREVIEWS = False
expected = [PREVIEW_RECORDS_DIR / f"preview_{i:02d}.pt" for i in range(5)]
if not REBUILD_PREVIEWS and all(path.is_file() for path in expected):
    PREVIEW_RECORD_PATHS = [str(path) for path in expected]
    print("preview records already exist ->", PREVIEW_RECORDS_DIR)
    print("preview count ->", len(PREVIEW_RECORD_PATHS))
else:
    local_shards = sorted(WDS_STATS_DIR.glob("train/*.tar"))
    if not local_shards:
        raise FileNotFoundError(
            f"No local shards under {WDS_STATS_DIR}/train — run the download cell first."
        )

    PREVIEW_RECORDS_DIR.mkdir(parents=True, exist_ok=True)
    tar = local_shards[0]
    dataset = wds.WebDataset(str(tar), shardshuffle=False).decode().shuffle(1000)

    preview_paths = []
    for index, sample in zip(range(5), dataset):
        record = sample_to_record(sample)
        out_path = PREVIEW_RECORDS_DIR / f"preview_{index:02d}.pt"
        torch.save(
            {
                "schema_version": 2,
                "image_latent": record.image_latent,
                "text_embedding": record.text_embedding,
                "attention_mask": record.attention_mask,
                "caption": record.caption,
                "metadata": record.metadata,
            },
            out_path,
        )
        preview_paths.append(out_path)
        print(f"saved preview_{index:02d} ->", out_path)
        print("caption:", record.caption[:120])

    PREVIEW_RECORD_PATHS = [str(path) for path in preview_paths]
    print("preview count ->", len(PREVIEW_RECORD_PATHS))



In [ ]:
%cd /content/Boomer

# Sanity-check the exact BoomerFLADiT config before launching a long training run.
!python tools/inspect_model.py \
  --model-type boomer_fla \
  --hidden-dim 1152 \
  --depth 28 \
  --num-heads 16 \
  --mlp-ratio 2.5 \
  --mixer-type fla_gated_deltanet \
  --fla-mode chunk \
  --use-short-conv \
  --conv-size 4 \
  --image-attn-every 4 \
  --image-attn-backend sdpa \
  --image-rope \
  --image-rope-theta 10000 \
  --cross-attn-backend sdpa \
  --cross-attn-qk-norm \
  --y-norm \
  --dtype bf16 \
  --forward

In [ ]:
import torch

# Explicit TF32 for fp32 paths (EMA shadow, fp32 loss). Main forward/backward stays bf16.
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print(
        "cuda matmul tf32:",
        torch.backends.cuda.matmul.allow_tf32,
        "| cudnn tf32:",
        torch.backends.cudnn.allow_tf32,
    )
else:
    print("no cuda — tf32 n/a")

In [ ]:
%cd /content/Boomer

import os

# Quiet HF shard downloads; step timing is printed by the trainer itself.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

print("checkpoints ->", CKPT_DIR)
print("previews    ->", PREVIEW_DIR)

# Fresh run: adaLN-Zero + 2D RoPE are not compatible with older non-rope checkpoints.
# To resume, add after the python line:
#   --resume "/content/drive/MyDrive/BoomerMain/checkpoints/boomer_fla_journeydb_rope/latent_dit_step_004000.pt" \

!python -u tools/overfit_latent_dit.py \
  --cache-wds "hf://datasets/akrao9/JourneyDBCache/train/*.tar" \
  --wds-local-dir {WDS_ROLL_DIR} \
  --wds-rolling-shards 25 \
  --wds-shuffle-buffer 1000 \
  --latent-stats-json {LATENT_STATS} \
  --latent-stats-mode channel \
  --model-type boomer_fla \
  --hidden-dim 1152 \
  --depth 28 \
  --num-heads 16 \
  --mlp-ratio 2.5 \
  --mixer-type fla_gated_deltanet \
  --fla-mode chunk \
  --use-short-conv \
  --conv-size 4 \
  --image-attn-every 4 \
  --image-attn-backend sdpa \
  --image-rope \
  --image-rope-theta 10000 \
  --cross-attn-backend sdpa \
  --cross-attn-qk-norm \
  --batch-size 80 \
  --num-workers 0 \
  --steps 100000 \
  --lr 1e-4 \
  --lr-scheduler linear-warmup-cosine \
  --warmup-steps 1000 \
  --min-lr-ratio 0.1 \
  --dtype bf16 \
  --flow-shift 3.0 \
  --t-sampler logit-normal \
  --timestep-weight \
  --y-norm \
  --ema-decay 0.96 \
  --ema-update-every 8 \
  --grad-clip 1.0 \
  --cond-dropout 0.1 \
  --fused-adamw \
  --log-every 50 \
  --log-csv {LOG_CSV} \
  --save-every 4000 \
  --checkpoint-dir {CKPT_DIR} \
  --preview-every 4000 \
  --preview-out-dir {PREVIEW_DIR} \
  --preview-cache-record {PREVIEW_RECORDS_DIR}/preview_00.pt \
  --preview-cache-record {PREVIEW_RECORDS_DIR}/preview_01.pt \
  --preview-cache-record {PREVIEW_RECORDS_DIR}/preview_02.pt \
  --preview-cache-record {PREVIEW_RECORDS_DIR}/preview_03.pt \
  --preview-cache-record {PREVIEW_RECORDS_DIR}/preview_04.pt \
  --preview-steps 16 \
  --preview-sampler stork2 \
  --preview-stork-derivative-order 1 \
  --preview-stork-substeps 5 \
  --seed 0 \
  --wandb \
  --wandb-project boomer \
  --wandb-run-name boomer_fla_journeydb_rope \
  --hf-token true



## Drive outputs (adaLN-Zero + 2D RoPE run)

- **Checkpoints:** `BoomerMain/checkpoints/boomer_fla_journeydb_rope/` (`.pt` resume + `.safetensors` inference)
- **Previews:** `BoomerMain/previews/boomer_fla_journeydb_rope/` (5 samples, STORK2 denoise every 4k steps)
- **Loss CSV:** `BoomerMain/logs/boomer_fla_journeydb_rope_loss.csv`
- **Stats JSON:** `BoomerMain/stats/journeydb_latent_stats.json` (~50k local shard samples)

Run order after setup: **download shards → stats → preview records → inspect model → TF32 → train**.

Run the STORK install cell before training. First start downloads 25 rolling shards (~50GB) into `/content/wds_roll`.
Drop `--fused-adamw` if step time regresses vs plain AdamW.

